# Soil water and soil CO2 columns

Reads the `met_forcing_rxn` MIN3P runs off S3 and writes five zarr stores back to
`s3://carbonplan-carbon-removal/ew-workflows-data/min3p/simulations/postprocessed/met_forcing_rxn/`:

| store | dims | time axis |
|---|---|---|
| `soilwater_co2_hourly.zarr` | `(site, time, depth, treat)` | 87 600 hours |
| `soilwater_co2_daily.zarr` | `(site, time, depth, treat)` | 3 650 days |
| `soilwater_co2_monthly.zarr` | `(site, time, depth, treat)` | 120 calendar months |
| `soilwater_co2_longterm.zarr` | `(site, time, depth, treat)` | 10 years |
| `soilwater_co2_profile_mean.zarr` | `(site, case, depth, treat)` | the 10-year record mean |

Each case's met forcing resolution (where a case refers to the met forcing) is stored as the time 
dimension from the run's `<case>.bcvs`. `case` is a dimension in the time-mean column zarr. 

## 0. Setup

In [1]:
import os
import re
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import s3fs
import xarray as xr

from byte_util.util import all_sites, states_per_site

# ----------------------------------------------------------------- knobs
WRITE = True          # False builds everything in memory but writes nothing to S3
OVERWRITE = True      # mode="w" on the zarr stores; False refuses to clobber
WORKERS = 8           # parallel S3 downloads
# ------------------------------------------------------------------------

SITES = list(all_sites)                              # all eight
CASES = ["hourly", "daily", "monthly", "longterm"]   # ordered coarse-ward
TREATMENTS = ["ctrl", "erw"]
RUN_DAYS = 3650
LONGTERM_BIN_DAYS = 365      # see the header: a choice, not a schedule read from the deck

BUCKET_ROOT = ("carbonplan-carbon-removal/ew-workflows-data/min3p/simulations"
               "/met_forcing_rxn/min3p_runs")
OUT_ROOT = ("s3://carbonplan-carbon-removal/ew-workflows-data/min3p/simulations"
            "/postprocessed/met_forcing_rxn")

# A *fresh* staging directory. The runs on S3 were re-run after `~/.cache/min3p_outputs/
# met_forcing_rxn` was populated -- that cache holds the old 13-monitor decks -- so nothing
# is read from it. Files here are validated against the S3 object size on every pass, so a
# re-upload upstream re-downloads rather than being silently reused.
CACHE = Path(os.environ.get("MIN3P_OUTPUT_CACHE", Path.home() / ".cache" / "min3p_outputs"))
STAGE = CACHE / "met_forcing_rxn_s3"
STAGE.mkdir(parents=True, exist_ok=True)

RUNS = [(s, c, t) for s in SITES for c in CASES for t in TREATMENTS]
fs = s3fs.S3FileSystem()

print(f"{len(RUNS)} runs: {len(SITES)} sites x {len(CASES)} cases x {len(TREATMENTS)} treatments")
print("staging into:", STAGE)

64 runs: 8 sites x 4 cases x 2 treatments
staging into: /Users/tylerkukla/.cache/min3p_outputs/met_forcing_rxn_s3


## 1. Staging

Four families are pulled per run: `<case>_N.gbp` (flow, carries `theta_a`), `<case>_N.gbg` (gas,
carries `co2(g)`), `<case>_o.gen` (the `ivol → z` table and the exit status) and `<case>.bcvs` (the
transient boundary schedule). 

`_o.gen` is checked for `normal exit`, so we catch any truncated runs.

In [2]:
def run_key(site, case, treat):
    return f"{BUCKET_ROOT}/{site}/{case}_{treat}"


def run_dir(site, case, treat):
    return STAGE / site / f"{case}_{treat}"


def wanted(name, case):
    """The file families this notebook needs, by name."""
    return (name in (f"{case}_o.gen", f"{case}.bcvs")
            or re.fullmatch(rf"{re.escape(case)}_\d+\.(gbp|gbg)", name) is not None)


def fetch_run(site, case, treat):
    """Stage one run; return (bytes fetched, monitor files present). Size-checked against S3."""
    local = run_dir(site, case, treat)
    local.mkdir(parents=True, exist_ok=True)
    got, n = 0, 0
    for info in fs.ls(run_key(site, case, treat), detail=True):
        name = info["name"].rsplit("/", 1)[-1]
        if not wanted(name, case):
            continue
        n += name.endswith((".gbp", ".gbg"))
        dest = local / name
        if dest.exists() and dest.stat().st_size == info["size"]:
            continue
        fs.get_file(info["name"], str(dest))
        got += info["size"]
    return got, n


with ThreadPoolExecutor(WORKERS) as pool:
    results = list(pool.map(lambda r: fetch_run(*r), RUNS))

fetched = sum(g for g, _ in results)
counts = {n for _, n in results}
print(f"{fetched / 1e9:.2f} GB fetched this pass; {counts} monitor files per run")
assert len(counts) == 1, f"runs disagree on how many monitor files they have: {counts}"

0.00 GB fetched this pass; {42} monitor files per run


In [3]:
def exit_status(site, case, treat):
    """The last banner MIN3P wrote, from the tail of `<case>_o.gen`."""
    text = (run_dir(site, case, treat) / f"{case}_o.gen").read_text(errors="replace")[-2000:]
    if "normal exit" in text:
        return "normal exit"
    fail = re.search(r"\*+\s*(.*?exit.*?|failure.*?)\s*\*+", text, re.I)
    return fail.group(1).strip() if fail else "unknown"


bad = {r: s for r in RUNS if (s := exit_status(*r)) != "normal exit"}
assert not bad, f"{len(bad)} runs did not exit normally: {bad}"
print(f"all {len(RUNS)} runs exited normally")

all 64 runs exited normally


## 2. Reading a run

In [4]:
_IVOL_RE = re.compile(r"ivol\s*=\s*(\d+).*?z\s*=\s*([0-9.eEdD+-]+)\s*m")
_NZ_RE = re.compile(r"control volumes in z-direction\s*=\s*(\d+)")


def read_transient(path):
    """A MIN3P transient output file as a DataFrame, with `attrs['volume']` set."""
    path = Path(path)
    head = []
    with path.open(errors="replace") as f:
        for _ in range(10):
            line = f.readline()
            if not line:
                break
            head.append(line)
            if line.lstrip().lower().startswith("zone"):
                break
    if not head or not head[-1].lstrip().lower().startswith("zone"):
        raise ValueError(f"no zone line in the first 10 lines of {path}")
    v0 = next(i for i, ln in enumerate(head) if ln.lstrip().lower().startswith("variables"))
    names = [n.strip() for n in re.findall(r'"([^"]*)"', "".join(head[v0:-1]))]
    df = pd.read_csv(path, sep=r"\s+", skiprows=len(head), header=None, engine="c")
    if df.shape[1] != len(names):
        raise ValueError(f"{path}: {len(names)} variable names, {df.shape[1]} columns")
    df.columns = names
    vol = re.search(r"volume\s*=\s*(\d+)", head[-1])
    if vol is None:
        raise ValueError(f"{path} has no control volume in its zone line")
    df.attrs["volume"] = int(vol.group(1))
    return df


def monitor_depths(gen_path):
    """`{ivol: depth below ground in cm}` for one run, read from its `<case>_o.gen`."""
    text = Path(gen_path).read_text(errors="replace")
    block = (text.split("control volume numbers and spatial locations:")[1]
             .split("output in terms")[0])
    ivol, z = np.array([[float(a), float(b)] for a, b in _IVOL_RE.findall(block)]).T
    nz = int(_NZ_RE.search(text).group(1))
    slope, intercept = np.polyfit(ivol, z, 1)
    depth = (slope * nz + intercept - z) * 100
    assert np.allclose(depth, np.round(depth), atol=1e-6), "monitors are not on whole cm"
    return dict(zip(ivol.astype(int).tolist(), np.round(depth).astype(int).tolist()))


def read_monitors(site, case, treat, ext):
    """Every `<case>_N.<ext>` monitor of one run, keyed by depth below ground in cm."""
    folder = run_dir(site, case, treat)
    depth_of = monitor_depths(folder / f"{case}_o.gen")
    out = {}
    for path in folder.glob(f"{case}_*.{ext}"):
        if not re.fullmatch(rf"{re.escape(case)}_\d+\.{ext}", path.name):
            continue
        df = read_transient(path)
        out[depth_of[df.attrs["volume"]]] = df
    return dict(sorted(out.items()))


# The depth axis read from runs
DEPTHS = list(read_monitors(SITES[0], "longterm", "ctrl", "gbg"))
for site, case, treat in RUNS:
    got = sorted(monitor_depths(run_dir(site, case, treat) / f"{case}_o.gen").values())
    assert got == DEPTHS, f"{site}/{case}_{treat} monitors {got}, not {DEPTHS}"
print(f"{len(DEPTHS)} monitored depths (cm):", DEPTHS)

21 monitored depths (cm): [0, 1, 5, 10, 20, 30, 40, 50, 51, 70, 81, 100, 101, 120, 130, 140, 150, 200, 201, 300, 301]


## 3. The time axis of each case

In [5]:
def forcing_edges(site, case, treat):
    """Bin edges in days for one run, read from its own transient boundary schedule."""
    path = run_dir(site, case, treat) / f"{case}.bcvs"
    if not path.exists():
        # No transient boundary block: the flux is a single constant for the whole run, so
        # there is no schedule to read and the annual binning is this notebook's choice.
        return np.arange(0, RUN_DAYS + 1, LONGTERM_BIN_DAYS, dtype=float)
    t = pd.read_csv(path, sep=r"\s+", header=None, usecols=[0]).to_numpy().ravel()
    t = np.round(np.unique(t) * 24) / 24          # snap to exact hours
    # The end of the run always closes the last bin. 
    return np.concatenate([[0.0], t[(t > 0) & (t < RUN_DAYS)], [float(RUN_DAYS)]])


# One schedule per case: the met record is site-specific but the *schedule* is not, so every
# run of a case must agree. Checking that is free and catches a mis-staged deck.
EDGES = {}
for case in CASES:
    schedules = {}
    for site in SITES:
        for treat in TREATMENTS:
            schedules.setdefault(forcing_edges(site, case, treat).tobytes(), []).append(
                f"{site}/{case}_{treat}")
    assert len(schedules) == 1, f"{case}: runs disagree on the forcing schedule, {schedules.keys()}"
    EDGES[case] = forcing_edges(SITES[0], case, TREATMENTS[0])

for case in CASES:
    e = EDGES[case]
    assert e[0] == 0 and e[-1] == RUN_DAYS, f"{case} edges span {e[0]}-{e[-1]}, not 0-{RUN_DAYS}"
    w = np.diff(e)
    print(f"{case:9s} {len(w):6d} bins   width {w.min():8.4f} - {w.max():8.4f} d   "
          f"first edges {np.round(e[1:4], 4)}")

hourly     87600 bins   width   0.0417 -   0.0417 d   first edges [0.0417 0.0833 0.125 ]
daily       3650 bins   width   1.0000 -   1.0000 d   first edges [1. 2. 3.]
monthly      120 bins   width  28.0000 -  31.0000 d   first edges [30. 61. 92.]
longterm      10 bins   width 365.0000 - 365.0000 d   first edges [ 365.  730. 1095.]


## 4. Build one dataset per case

In [6]:
TMID = (np.arange(RUN_DAYS * 24) + 0.5) / 24.0     # hour midpoints, in days

VARS = {                     # name -> (extension, column, units, long_name)
    "theta": ("gbp", "theta_a", "m3 m-3", "volumetric soil water content"),
    "pco2": ("gbg", "co2(g)", "atm", "soil CO2 partial pressure"),
}


def dedupe(t, v):
    """One monitor's raw record, with repeated timestamps dropped."""
    t = np.asarray(t, dtype=float)
    keep = np.ones(t.size, dtype=bool)
    keep[:-1] = t[1:] != t[:-1]          # MIN3P repeats a timestamp at forced output times
    return t[keep], np.asarray(v, dtype=float)[keep]


def on_hour_grid(t, v):
    """A de-duplicated record interpolated onto the hour midpoints."""
    grid = np.interp(TMID, t, v)
    grid[TMID > t[-1]] = np.nan          # never extrapolate past the end of a run
    return grid


def record_mean(t, v):
    """The exact time average of the raw record, at its own adaptive resolution.

    MIN3P writes the state at instants, and linear interpolation between them is the
    same reconstruction `np.interp` uses, so the trapezoid rule over the raw samples is
    the exact mean of that reconstruction -- every step weighted by its own duration,
    nothing resampled, nothing dropped.
    """
    if t[-1] < RUN_DAYS - 1e-6:
        return np.nan                    # a truncated run has no 10-year mean
    return np.trapezoid(v, t) / (t[-1] - t[0])


def binning(edges):
    """(start index, sample count) per bin, for a grid of hour midpoints."""
    counts = np.round(np.diff(edges) * 24).astype(int)
    assert counts.min() >= 1, "a forcing step is shorter than an hour"
    assert counts.sum() == TMID.size, f"{counts.sum()} hours binned, {TMID.size} in the run"
    return np.concatenate([[0], np.cumsum(counts)[:-1]]), counts


def build_case(case):
    """`(site, time, depth, treat)` bin means, and the record mean off the hourly grid."""
    edges = EDGES[case]
    starts, counts = binning(edges)
    shape = (len(SITES), len(counts), len(DEPTHS), len(TREATMENTS))
    binned = {name: np.full(shape, np.nan, dtype=np.float32) for name in VARS}
    means = {name: np.full((len(SITES), len(DEPTHS), len(TREATMENTS)), np.nan, dtype=np.float32)
             for name in VARS}

    for i, site in enumerate(SITES):
        for k, treat in enumerate(TREATMENTS):
            families = {ext: read_monitors(site, case, treat, ext)
                        for ext in {e for e, *_ in VARS.values()}}
            for name, (ext, col, *_) in VARS.items():
                for j, depth in enumerate(DEPTHS):
                    df = families[ext][depth]
                    t, v = dedupe(df["time"].to_numpy(), df[col].to_numpy())
                    grid = on_hour_grid(t, v)
                    binned[name][i, :, j, k] = np.add.reduceat(grid, starts) / counts
                    means[name][i, j, k] = record_mean(t, v)
        print(f"  {case:9s} {site:14s} done", flush=True)

    dims = ("site", "time", "depth", "treat")
    coords = {"site": SITES, "time": (edges[:-1] + edges[1:]) / 2,
              "depth": np.array(DEPTHS, dtype=np.int16), "treat": TREATMENTS}
    ds = xr.Dataset(
        {name: (dims, arr, {"units": VARS[name][2],
                            "long_name": f"{VARS[name][3]}, {case}-bin mean",
                            "cell_methods": "time: mean"})
         for name, arr in binned.items()}, coords=coords)
    ds = ds.assign_coords(time_bnds=(("time", "bnds"),
                                     np.stack([edges[:-1], edges[1:]], axis=1)))
    # NOT a CF time axis: "<units> since <date>" would make xarray try to decode this against a
    # calendar on every open, and there is no calendar date behind these runs.
    ds["time"].attrs = {"units": "days", "long_name": "simulation time",
                        "bounds": "time_bnds",
                        "description": ("midpoint of each forcing bin, in days elapsed since "
                                        "the start of the simulation")}
    ds["time_bnds"].attrs = {"units": "days", "long_name": "forcing bin edges"}
    ds["site"].attrs = {"long_name": "soil series"}
    ds["depth"].attrs = {"units": "cm", "long_name": "depth below ground surface",
                         "positive": "down"}
    ds["treat"].attrs = {"long_name": "treatment",
                         "description": "ctrl = unamended; erw = forsterite amendment"}
    ds = ds.assign_coords(state=("site", [states_per_site[s] for s in SITES]))
    schedule = ("annual bins; this case has no transient boundary block, so the binning is a "
                "choice of this notebook and not a schedule read from the deck"
                if case == "longterm" else
                f"bin edges are the forcing steps of {case}.bcvs")
    ds.attrs = {
        "title": f"MIN3P met_forcing_rxn soil water and soil CO2, {case} forcing",
        "case": case,
        "description": ("Bin-mean volumetric water content and soil CO2 partial pressure at every "
                        "monitored depth of the met_forcing_rxn MIN3P runs, on this case's own "
                        "meteorological forcing schedule."),
        # "source": f"s3://{BUCKET_ROOT}/<site>/{case}_<treat>/",
        "time_axis": schedule,
        "method": ("records de-duplicated and interpolated onto a uniform grid of hour midpoints, "
                   "then averaged within each forcing bin; theta_a is the water content per "
                   "<case>_o.fls"),
        "created_by": "figures/postprocess-data/create-soilwater+co2-columns.ipynb",
    }

    # Named for what it is, and *not* `record_mean` -- that is the function above, and a local
    # of the same name would shadow it for the whole body.
    means_ds = xr.Dataset(
        {name: (("site", "depth", "treat"), arr) for name, arr in means.items()},
        coords={"site": SITES, "depth": np.array(DEPTHS, dtype=np.int16), "treat": TREATMENTS})
    return ds, means_ds


columns, record_means = {}, {}
for case in CASES:
    columns[case], record_means[case] = build_case(case)
    print(f"{case}: {dict(columns[case].sizes)}", flush=True)

  hourly    Cecil          done
  hourly    Flanagan       done
  hourly    HoustonBlack   done
  hourly    Kalamazoo      done
  hourly    Kuma           done
  hourly    Palouse        done
  hourly    Pullman        done
  hourly    Yolo           done
hourly: {'site': 8, 'time': 87600, 'depth': 21, 'treat': 2, 'bnds': 2}
  daily     Cecil          done
  daily     Flanagan       done
  daily     HoustonBlack   done
  daily     Kalamazoo      done
  daily     Kuma           done
  daily     Palouse        done
  daily     Pullman        done
  daily     Yolo           done
daily: {'site': 8, 'time': 3650, 'depth': 21, 'treat': 2, 'bnds': 2}
  monthly   Cecil          done
  monthly   Flanagan       done
  monthly   HoustonBlack   done
  monthly   Kalamazoo      done
  monthly   Kuma           done
  monthly   Palouse        done
  monthly   Pullman        done
  monthly   Yolo           done
monthly: {'site': 8, 'time': 120, 'depth': 21, 'treat': 2, 'bnds': 2}
  longterm  Cecil     

In [7]:
# sanity check before writing. water content sits strictly inside [0, 1], soil CO2 is
# positive, and no unexpected NaNs. 
for case, ds in columns.items():
    for name in VARS:
        assert ds[name].notnull().all(), f"{case}/{name} has missing bins"
    lo, hi = float(ds["theta"].min()), float(ds["theta"].max())
    assert 0 < lo and hi < 1, f"{case} water content out of range: {lo}-{hi}"
    assert float(ds["pco2"].min()) > 0, f"{case} has non-positive pCO2"
    # The bins and the record mean are now independent estimates of the same integral -- the
    # bins go through the hourly grid, the record mean does not -- so agreement between them
    # is a real check on the binning rather than an identity. Weighting by bin width is what
    # makes the monthly case comparable at all.
    w = xr.DataArray(np.diff(EDGES[case]), dims="time")
    gap = max(float(abs(ds[name].weighted(w).mean("time") - record_means[case][name]).max()
                    / abs(record_means[case][name]).max()) for name in VARS)
    assert gap < 1e-3, f"{case}: bins and raw record mean disagree by {gap:.2e}"
    print(f"{case:9s} {ds.sizes['time']:6d} bins   theta {lo:.3f}-{hi:.3f} m3/m3   "
          f"pco2 {float(ds['pco2'].min()):.2e}-{float(ds['pco2'].max()):.2e} atm   "
          f"bins vs raw record mean {gap:.1e}")

columns["monthly"]

hourly     87600 bins   theta 0.067-0.526 m3/m3   pco2 1.57e-06-5.44e-02 atm   bins vs raw record mean 1.7e-05
daily       3650 bins   theta 0.067-0.518 m3/m3   pco2 1.76e-06-5.42e-02 atm   bins vs raw record mean 6.5e-07
monthly      120 bins   theta 0.070-0.494 m3/m3   pco2 2.99e-06-5.67e-02 atm   bins vs raw record mean 4.3e-08
longterm      10 bins   theta 0.083-0.480 m3/m3   pco2 1.66e-05-5.21e-02 atm   bins vs raw record mean 4.3e-08


<xarray.Dataset> Size: 326kB
Dimensions:    (site: 8, time: 120, depth: 21, treat: 2, bnds: 2)
Coordinates:
  * site       (site) <U12 384B 'Cecil' 'Flanagan' ... 'Pullman' 'Yolo'
    state      (site) <U11 352B 'Northern SC' 'Central IL' ... 'Central CA'
  * time       (time) float64 960B 15.0 45.5 76.5 ... 3.606e+03 3.636e+03
  * depth      (depth) int16 42B 0 1 5 10 20 30 40 ... 140 150 200 201 300 301
  * treat      (treat) <U4 32B 'ctrl' 'erw'
    time_bnds  (time, bnds) float64 2kB 0.0 30.0 30.0 ... 3.621e+03 3.65e+03
Dimensions without coordinates: bnds
Data variables:
    theta      (site, time, depth, treat) float32 161kB 0.2874 0.2874 ... 0.3975
    pco2       (site, time, depth, treat) float32 161kB 0.0004494 ... 0.002307
Attributes:
    title:        MIN3P met_forcing_rxn soil water and soil CO2, monthly forcing
    case:         monthly
    description:  Bin-mean volumetric water content and soil CO2 partial pres...
    time_axis:    bin edges are the forcing steps of monthly.bcvs
    method:       records de-duplicated and interpolated onto a uniform grid ...
    created_by:   figures/postprocess-data/create-soilwater+co2-columns.ipynb

## 5. Taking the record mean

Time-mean for each case + site. 

In [8]:
profile_mean = (xr.concat([record_means[c] for c in CASES],
                          dim=pd.Index(CASES, name="case"))
                .transpose("site", "case", "depth", "treat"))
profile_mean = profile_mean.assign_coords(state=("site", [states_per_site[s] for s in SITES]))

for name in VARS:
    profile_mean[name].attrs = {"units": VARS[name][2],
                                "long_name": f"{VARS[name][3]}, 10-year record mean",
                                "cell_methods": "time: mean (interval: 10 years)"}
profile_mean["case"].attrs = {
    "long_name": "meteorological forcing resolution",
    "description": ("resolution the 10-year met record was resampled to before it was handed to "
                    "MIN3P; ordered coarse-ward, hourly is the reference"),
}
profile_mean["site"].attrs = {"long_name": "soil series"}
profile_mean["depth"].attrs = {"units": "cm", "long_name": "depth below ground surface",
                               "positive": "down"}
profile_mean["treat"].attrs = {"long_name": "treatment",
                               "description": "ctrl = unamended; erw = forsterite amendment"}
profile_mean.attrs = {
    "title": "MIN3P met_forcing_rxn mean soil water and soil CO2 profiles",
    "description": ("Ten-year record mean of volumetric water content and soil CO2 partial "
                    "pressure at every monitored depth, for each site, forcing resolution and "
                    "treatment."),
    # "source": f"s3://{BUCKET_ROOT}/",
    "method": ("trapezoid-rule time integral of the raw adaptive-timestep MIN3P record divided by "
               "the record length; independent of the binning used by the per-case stores"),
    "created_by": "figures/postprocess-data/create-soilwater+co2-columns.ipynb",
}

print(dict(profile_mean.sizes))
profile_mean

{'site': 8, 'case': 4, 'depth': 21, 'treat': 2}


<xarray.Dataset> Size: 12kB
Dimensions:  (site: 8, case: 4, depth: 21, treat: 2)
Coordinates:
  * site     (site) <U12 384B 'Cecil' 'Flanagan' ... 'Pullman' 'Yolo'
    state    (site) <U11 352B 'Northern SC' 'Central IL' ... 'Central CA'
  * case     (case) StringDType(na_object=nan) 64B 'hourly' ... 'longterm'
  * depth    (depth) int16 42B 0 1 5 10 20 30 40 ... 130 140 150 200 201 300 301
  * treat    (treat) <U4 32B 'ctrl' 'erw'
Data variables:
    theta    (site, case, depth, treat) float32 5kB 0.2801 0.2801 ... 0.3983
    pco2     (site, case, depth, treat) float32 5kB 0.0004367 ... 0.00233
Attributes:
    title:        MIN3P met_forcing_rxn mean soil water and soil CO2 profiles
    description:  Ten-year record mean of volumetric water content and soil C...
    method:       trapezoid-rule time integral of the raw adaptive-timestep M...
    created_by:   figures/postprocess-data/create-soilwater+co2-columns.ipynb

## 6. Write

In [9]:
STORES = {f"soilwater_co2_{case}": ds for case, ds in columns.items()}
STORES["soilwater_co2_profile_mean"] = profile_mean

CHUNK_LIMIT = 8e6      # bytes per chunk, uncompressed, above which time is split by year


def chunks_for(ds):
    chunks = {d: n for d, n in ds.sizes.items()}
    per_site = ds.sizes.get("time", 1) * ds.sizes["depth"] * ds.sizes["treat"] * 4
    if per_site > CHUNK_LIMIT:
        chunks["site"], chunks["time"] = 1, 24 * 365
    return chunks


def write(name, ds):
    url = f"{OUT_ROOT}/{name}.zarr"
    if not OVERWRITE and fs.exists(url.replace("s3://", "")):
        raise FileExistsError(f"{url} exists and OVERWRITE is False")
    ds.chunk(chunks_for(ds)).to_zarr(url, mode="w", zarr_format=2, consolidated=True)
    return url


if WRITE:
    for name, ds in STORES.items():
        print(f"wrote {write(name, ds)}   chunks {chunks_for(ds)}", flush=True)
else:
    print("WRITE is False -- nothing written")
    for name, ds in STORES.items():
        print(f"  would write {OUT_ROOT}/{name}.zarr  {dict(ds.sizes)}  {chunks_for(ds)}")

wrote s3://carbonplan-carbon-removal/ew-workflows-data/min3p/simulations/postprocessed/met_forcing_rxn/soilwater_co2_hourly.zarr   chunks {'site': 1, 'time': 8760, 'depth': 21, 'treat': 2, 'bnds': 2}
wrote s3://carbonplan-carbon-removal/ew-workflows-data/min3p/simulations/postprocessed/met_forcing_rxn/soilwater_co2_daily.zarr   chunks {'site': 8, 'time': 3650, 'depth': 21, 'treat': 2, 'bnds': 2}
wrote s3://carbonplan-carbon-removal/ew-workflows-data/min3p/simulations/postprocessed/met_forcing_rxn/soilwater_co2_monthly.zarr   chunks {'site': 8, 'time': 120, 'depth': 21, 'treat': 2, 'bnds': 2}
wrote s3://carbonplan-carbon-removal/ew-workflows-data/min3p/simulations/postprocessed/met_forcing_rxn/soilwater_co2_longterm.zarr   chunks {'site': 8, 'time': 10, 'depth': 21, 'treat': 2, 'bnds': 2}
wrote s3://carbonplan-carbon-removal/ew-workflows-data/min3p/simulations/postprocessed/met_forcing_rxn/soilwater_co2_profile_mean.zarr   chunks {'site': 8, 'case': 4, 'depth': 21, 'treat': 2}


## 7. Read back

In [10]:
if WRITE:
    for name, ds in STORES.items():
        url = f"{OUT_ROOT}/{name}.zarr"
        back = xr.open_zarr(url).load()
        xr.testing.assert_allclose(back, ds)
        mb = sum(f["size"] for f in fs.find(url.replace("s3://", ""), detail=True).values()) / 1e6
        print(f"{name:34s} {str(dict(back.sizes)):<62s} {mb:6.1f} MB")
    print("\nall stores round-trip")

soilwater_co2_hourly               {'site': 8, 'time': 87600, 'depth': 21, 'treat': 2, 'bnds': 2}  140.0 MB
soilwater_co2_daily                {'site': 8, 'time': 3650, 'depth': 21, 'treat': 2, 'bnds': 2}     7.0 MB
soilwater_co2_monthly              {'site': 8, 'time': 120, 'depth': 21, 'treat': 2, 'bnds': 2}      0.2 MB
soilwater_co2_longterm             {'site': 8, 'time': 10, 'depth': 21, 'treat': 2, 'bnds': 2}       0.0 MB
soilwater_co2_profile_mean         {'site': 8, 'case': 4, 'depth': 21, 'treat': 2}                   0.0 MB

all stores round-trip


In [ ]:
# ---